In [ ]:
from math import floor
from typing import NamedTuple

## Golden Model for NMS Algorithm

Uses File I/O Golden Model for hardware verification inspired from this [article](https://thedatabus.in/python-iverilog-verification/)

The input were considering has these parameters
- The confidence score `c`
- Lower left cordinate `(x, y)`
- Upper right coordinate `(a, b)`

In [ ]:
x1, y1, a1, b1, c1 = 200, 300, 400, 500, 0.86  # point 1
x2, y2, a2, b2, c2 = 350, 350, 450, 600, 0.65  # point 2

The 12 initial representation of coodinates comes from UART bit alignment. Take 12 bit for granted and do the calculations

Bounding box rejection criteria
$$\text{Area\_Overlap} \times 2^k \geq \text{Threshold\_INT} \times Area\_Union$$

| variable | Signed / Unsigned | Integer Bits | Fractional Bits | Total Width | Domain max (1080p) | Representable Range |
| --- | --- | --- | --- | --- | --- | --- |
| x/y/a/b | u | 12 | 0 | 12 | 1920 | 0 to 4095 |
| area1/area2 | u | 24 | 0 | 24 | 2073600 | 0 to 16777215 |
| xx/yy/aa/bb | u | 12 | 0 | 12 | 1920 | 0 to 4095 |
| t_w/t_h | s | 13 | 0 | 13 | 1920 | -4096 to +4095
| w/h | u | 12 | 0 | 12 | 1920 | 0 to 4095 | # can clamp it back
| intersection_area | u | 24 | 0 | 24 | 2073600 | 0 to 16777215 |
|  t_union_area | s | 26 | 0 | 26 | 2073600 | -33554432 to 33554431 | # positive side dominates thus to account for sign, we need 26 bits (assumes that the 3 data paths are independent)
|  union_area | u | 25 | 0 | 25 | 2073600 | 0 to 33554431 | # can clamp it back to 25 bits
|  T_INT | u | 8 | 0 | 8 | 126 | 0 to 126 |
|  2^k | u | 8 | 0 | 8 | 126 | 0 to 126 | # Q0.8 precision is chosen for simplicity
|  LHS | u | 32 | 0 | 32 | 261273600 | 0 to 4294967296 |
|  RHS | u | 33 | 0 | 33 | 261273600 | 0 to 8,589,934,592 | # 33 bits would be fixed length to compare

For Tie breaking, we'll sort (confidence, index) total of 21 bit numbers.

In [ ]:
# Calculations needed for IoU

area1 = (a1 - x1) * (b1 - y1)
area2 = (a2 - x2) * (b2 - y2)

# maximize the lower left and minimize the upper right (nature of intersection)
xx = max(x1, x2)
yy = max(y1, y2)
aa = min(a1, a2)
bb = min(b1, b2)

# width and height
t_w = aa - xx
t_h = bb - yy
w = max(0, t_w)
h = max(0, t_h)

# intersection and union area
intersection_area = w * h
union_area = area1 + area2 - intersection_area
iou = intersection_area / union_area

In [ ]:
iou

In [ ]:
bbox1 = 200, 300, 400, 500, 0.85
bbox2 = 350, 350, 450, 600, 0.65

In [ ]:
def calculate_iou(bbox1: tuple, bbox2: tuple) -> float:
    """Calculate IoU value from bounding boxes.

    Assumes each bounding box is a `tuple` follows this structure
    `(lower left cordinate, upper right cordinate, confidence_score).

    Args:
        bbox1 (tuple): bounding box 1
        bbox2 (tuple): bounding box 2

    Returns:
        float: calulated IoU value
    """
    x1, y1, a1, b1, _ = bbox1
    x2, y2, a2, b2, _ = bbox2

    # Calculations needed for IoU
    area1 = (a1 - x1) * (b1 - y1)
    area2 = (a2 - x2) * (b2 - y2)

    # maximize the lower left and minimize the upper right (nature of intersection)
    xx = max(x1, x2)
    yy = max(y1, y2)
    aa = min(a1, a2)
    bb = min(b1, b2)

    # width and height
    w = max(0, (aa - xx))
    h = max(0, (bb - yy))

    # intersection and union area
    intersection_area = w * h
    union_area = area1 + area2 - intersection_area
    iou = intersection_area / union_area

    return iou

In [ ]:
def nms(boxes: list, iou_threshold: float) -> list:

    sorted_boxes = sorted(boxes, key=lambda box: box[4], reverse=True)
    valid = [True] * len(sorted_boxes)
    keep = []

    for i in range(len(sorted_boxes)):
        if valid[i]:
            keep.append(sorted_boxes[i])
            valid[i] = False

            for j in range(i + 1, len(sorted_boxes)):
                if valid[j]:
                    iou = calculate_iou(sorted_boxes[i], sorted_boxes[j])
                    if iou >= iou_threshold:
                        valid[j] = False

    return keep


In [ ]:
# (x1, y1, x2, y2, confidence)
test_boxes = [
    # Cluster A — cat detection (~8 overlapping boxes)
    (10, 10, 50, 50, 0.95),
    (12, 11, 52, 51, 0.90),
    (9, 13, 48, 53, 0.85),
    (14, 9, 54, 49, 0.70),
    (11, 14, 51, 54, 0.65),
    (15, 12, 55, 52, 0.55),
    (8, 8, 46, 46, 0.40),
    (13, 15, 53, 55, 0.30),
    # Cluster B — dog detection (~8 overlapping boxes)
    (100, 100, 150, 150, 0.92),
    (102, 101, 152, 151, 0.88),
    (98, 103, 148, 153, 0.80),
    (104, 99, 154, 149, 0.72),
    (101, 105, 151, 155, 0.60),
    (103, 98, 153, 148, 0.50),
    (97, 102, 147, 152, 0.42),
    (105, 104, 155, 154, 0.28),
    # Cluster C — car detection (~8 overlapping boxes)
    (200, 50, 260, 100, 0.93),
    (202, 52, 262, 102, 0.87),
    (198, 48, 258, 98, 0.78),
    (204, 53, 264, 103, 0.68),
    (201, 47, 261, 97, 0.58),
    (199, 55, 259, 105, 0.48),
    (203, 49, 263, 99, 0.38),
    (197, 51, 257, 101, 0.25),
    # Cluster D — person detection (~5 overlapping boxes)
    (50, 200, 100, 280, 0.91),
    (52, 202, 102, 282, 0.82),
    (48, 198, 98, 278, 0.73),
    (54, 203, 104, 283, 0.62),
    (47, 199, 97, 279, 0.45),
    # Isolated boxes — should all survive
    (300, 300, 340, 340, 0.75),
    (0, 280, 30, 310, 0.35),
    (280, 0, 320, 40, 0.20),
]

In [ ]:
nms(test_boxes, 0.5)

In [ ]:
T = 0.5
T_INT = min(floor(T * 256), 255)

In [ ]:
COORD_BITS, COORD_MAX = 12, 4095
CONF_BITS, CONF_SCALE = 16, 2**16 - 1
T_BITS, T_MAX, K = 8, 255, 8
IDX_BITS, IDX_MASK, MAX_BOXES = 5, 31, 32
KEY_BITS = CONF_BITS + IDX_BITS
LHS_WIDTH, RHS_WIDTH = 32, 33

# Intermediate signal widths (see docs/architecture.md). TW_BITS is signed,
# every other width here is unsigned.
AREA_BITS = 24
TW_BITS = 13
WH_BITS = 12
INTER_BITS = 24
UNION_BITS = 25

In [ ]:
def assert_unsigned(value: int, width: int, name: str) -> None:
    if 0 <= value < 2**width:
        return
    raise ValueError(name)

In [ ]:
def assert_signed(value: int, width: int, name: str) -> None:
    if -(2 ** (width - 1)) <= value < 2 ** (width - 1):
        return
    raise ValueError(name)

In [ ]:
def quantize_confidence(confidence: float) -> int:
    return floor(confidence * (2**16 - 1))

In [ ]:
def quantize_threshold(threshold: float) -> int:
    return min(floor(threshold * 256), 255)

In [ ]:
class IntBox(NamedTuple):
    x: int
    y: int
    a: int
    b: int
    conf: int
    index: int

In [ ]:
def to_int_boxes(boxes: list[tuple]) -> list[IntBox]:
    if len(boxes) > MAX_BOXES:
        msg = f"{len(boxes)} boxes exceeds the {MAX_BOXES} box capacity"
        raise ValueError(msg)

    result: list[IntBox] = []
    for i, (x, y, a, b, c) in enumerate(boxes):
        assert_unsigned(x, COORD_BITS, "x")
        assert_unsigned(y, COORD_BITS, "y")
        assert_unsigned(a, COORD_BITS, "a")
        assert_unsigned(b, COORD_BITS, "b")

        conf = quantize_confidence(c)
        assert_unsigned(conf, CONF_BITS, "confidence")

        result.append(IntBox(x, y, a, b, conf, i))
    return result

In [ ]:
def intersection_union(box1: IntBox, box2: IntBox) -> tuple[int, int]:
    x1, y1, a1, b1, _, _ = box1
    x2, y2, a2, b2, _, _ = box2

    # Calculations needed for IoU
    area1 = (a1 - x1) * (b1 - y1)
    area2 = (a2 - x2) * (b2 - y2)
    assert_unsigned(area1, AREA_BITS, "area1")
    assert_unsigned(area2, AREA_BITS, "area2")

    # maximize the lower left and minimize the upper right (nature of intersection)
    xx = max(x1, x2)
    yy = max(y1, y2)
    aa = min(a1, a2)
    bb = min(b1, b2)

    # width and height, signed before the clamp
    t_w = aa - xx
    t_h = bb - yy
    assert_signed(t_w, TW_BITS, "t_w")
    assert_signed(t_h, TW_BITS, "t_h")

    w = max(0, t_w)
    h = max(0, t_h)
    assert_unsigned(w, WH_BITS, "w")
    assert_unsigned(h, WH_BITS, "h")

    # intersection and union area
    intersection_area = w * h
    union_area = area1 + area2 - intersection_area
    assert_unsigned(intersection_area, INTER_BITS, "intersection_area")
    assert_unsigned(union_area, UNION_BITS, "union_area")

    return (intersection_area, union_area)

In [ ]:
def sort_key(box: IntBox) -> int:
    result = box.conf << 5 | (~box.index & 0x1F)
    assert_unsigned(result, KEY_BITS, "Sort key exceeds the width limit")
    return result

In [ ]:
def should_suppress(box1: IntBox, box2: IntBox, t_int: int) -> bool:
    intersection_area, union_area = intersection_union(box1, box2)
    if union_area == 0:
        return True

    lhs = intersection_area << K
    assert_unsigned(lhs, LHS_WIDTH, "lhs")
    rhs = union_area * t_int
    assert_unsigned(rhs, RHS_WIDTH, "rhs")
    return lhs >= rhs

In [ ]:
def nms_int(boxes: list[IntBox], t_int: int) -> list:

    sorted_boxes = sorted(boxes, key=sort_key, reverse=True)
    valid = [True] * len(sorted_boxes)
    keep = []

    for i in range(len(sorted_boxes)):
        if valid[i]:
            keep.append(sorted_boxes[i])
            valid[i] = False

            for j in range(i + 1, len(sorted_boxes)):
                if valid[j] and should_suppress(
                    sorted_boxes[i],
                    sorted_boxes[j],
                    t_int,
                ):
                    valid[j] = False

    return keep


In [ ]:
nms_int(to_int_boxes(test_boxes), quantize_threshold(0.5))